# B05A — Diagnosis Hinton KD sebelum evaluasi test

Membaca hasil B05 seed 42 dan cache Teacher yang **sudah ada**. Default memeriksa keempat dataset; fokus interpretasi adalah kegagalan INbreast.

**Jalankan di Colab CPU dengan Run all.** Tidak perlu GPU, instalasi TensorFlow, training ulang, atau ekstraksi citra. Notebook tidak membaca `test.csv`, prediksi test, maupun `all.csv`. Parameter training, threshold 0,5, checkpoint, dan protokol B05 tetap utuh.

Hasilnya adalah satu ZIP diagnosis. Pemeriksaan yang lulus bukan keputusan otomatis untuk membuka test. Metadata B05 mencatat bahwa test pernah diperiksa pada B03; hal ini tetap harus dinyatakan saat pelaporan akhir.

## Cell 1 — Google Drive dan konfigurasi
Pastikan folder Research sesuai Drive Anda. Empat ID eksperimen di bawah berasal dari laporan validation yang sudah diperiksa. Bila ingin hanya INbreast, ubah `DATASETS=['inbreast']`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive, files
from IPython.display import display
drive.mount('/content/drive')
RESEARCH=Path('/content/drive/MyDrive/Research')
DATASETS=['cbis_ddsm','cmmd','inbreast','mias']
EXPERIMENT_IDS={
    'cbis_ddsm':'46f81a244f81',
    'cmmd':'007a76eeecfb',
    'inbreast':'cbfc09b8c7cb',
    'mias':'f63a146b0869',
}
B05_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05_hinton_kd_v1'
PREPROCESSING_ROOT=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1'
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT=RESEARCH/'outputs/benchmark_4datasets/b05a_diagnostics'/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
DOWNLOAD_REPORT=True
print('Output diagnosis:',OUTPUT)
print('Tidak ada training atau evaluasi test dalam notebook ini.')


## Cell 2 — Fungsi diagnosis
Memeriksa hash sumber, hash cache, identitas sampel/label, bobot awal yang dipasangkan, perhitungan metrik, dan pemilihan checkpoint. File asli hanya dibaca. Jika cache atau data sumber hilang, pemeriksaan dataset tersebut dinyatakan belum lengkap dan error dimasukkan ke ZIP.

In [ ]:
import json, hashlib, zipfile, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score, confusion_matrix)

READ_LEDGER=[]
def read_bytes(path):
    path=Path(path)
    if path.name.lower().startswith('test') or path.name=='all.csv':
        raise RuntimeError('Diagnosis hanya boleh membaca train/validation: '+str(path))
    b=path.read_bytes()
    READ_LEDGER.append(dict(path=str(path),sha256=hashlib.sha256(b).hexdigest(),bytes=len(b)))
    return b

def read_json(path):return json.loads(read_bytes(path))

def read_csv(path,expected_hash=None):
    import io
    b=read_bytes(path)
    if expected_hash and hashlib.sha256(b).hexdigest()!=expected_hash:
        raise RuntimeError('Hash CSV tidak cocok: '+str(path))
    return pd.read_csv(io.BytesIO(b),float_precision='round_trip',keep_default_na=False,
                       dtype={'sample_id':str,'patient_id':str})

def save_json(path,obj):
    def safe(x):
        if isinstance(x,dict):return {str(k):safe(v) for k,v in x.items()}
        if isinstance(x,(list,tuple)):return [safe(v) for v in x]
        if isinstance(x,np.generic):return safe(x.item())
        if isinstance(x,float) and not np.isfinite(x):return None
        return x
    Path(path).write_text(json.dumps(safe(obj),indent=2,allow_nan=False),encoding='utf-8')

def mapped_path(path,research):
    old='/content/drive/MyDrive/Research'
    s=str(path)
    return Path(research)/s[len(old):].lstrip('/') if s==old or s.startswith(old+'/') else Path(s)

def check_table(frame):
    required={'sample_id','patient_id','label'}
    if not required.issubset(frame.columns):raise ValueError('Kolom identitas/label tidak lengkap')
    if frame.empty or frame.sample_id.duplicated().any():raise ValueError('ID kosong/duplikat')
    if (frame.sample_id=='').any() or (frame.patient_id=='').any():raise ValueError('Identitas kosong')
    if not frame.label.isin([0,1]).all():raise ValueError('Label bukan biner')

def align(frame,reference):
    check_table(frame);check_table(reference)
    if set(frame.sample_id)!=set(reference.sample_id):raise ValueError('Sampel tidak sama')
    result=frame.set_index('sample_id').loc[reference.sample_id].reset_index()
    for c in ['sample_id','patient_id','label']:
        if not np.array_equal(result[c].to_numpy(),reference[c].to_numpy()):
            raise ValueError('Identitas/label tidak cocok: '+c)
    return result

def sigmoid(z):
    z=np.asarray(z,dtype=np.float64)
    return np.exp(-np.logaddexp(0.,-z))

def measures(y,p):
    y=np.asarray(y);p=np.asarray(p,dtype=float)
    if len(y)!=len(p) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():raise ValueError('Probabilitas tidak valid')
    pred=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
        f1=f1_score(y,pred,zero_division=0),macro_f1=f1_score(y,pred,labels=[0,1],average='macro',zero_division=0),
        accuracy=accuracy_score(y,pred),balanced_accuracy=balanced_accuracy_score(y,pred),specificity=tn/(tn+fp),
        roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
        average_precision=average_precision_score(y,p),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

def gradient_terms(y,z_student,z_teacher,cw,alpha,temp):
    # Derivative wrt ONE scalar Student logit, before the batch-mean factor 1/N.
    # These are not gradients wrt neural-network parameters.
    w=np.array([cw[int(v)] for v in y])
    hard=(1-alpha)*w*(sigmoid(z_student)-y)
    soft=alpha*temp*(sigmoid(np.asarray(z_student)/temp)-sigmoid(np.asarray(z_teacher)/temp))
    return hard,soft

def distribution(values):
    a=np.asarray(values,dtype=float)
    return dict(n=len(a),minimum=float(a.min()),q10=float(np.quantile(a,.1)),median=float(np.median(a)),
                mean=float(a.mean()),q90=float(np.quantile(a,.9)),maximum=float(a.max()))

def ready_frames(protocol,preprocessing_root):
    candidates=[]
    for path in sorted(Path(preprocessing_root).rglob('READY.json')):
        meta=read_json(path)
        if meta.get('dataset')==protocol['dataset'] and meta.get('file_sha256')==protocol['data_sha256']:
            candidates.append(path)
    if len(candidates)!=1:raise FileNotFoundError('Butuh satu READY.json B04A yang sesuai. Ditemukan: '+str(candidates))
    parent=candidates[0].parent
    frames={s:read_csv(parent/(s+'.csv'),protocol['data_sha256'][s+'.csv']) for s in ['train','validation']}
    for frame in frames.values():check_table(frame)
    if set(frames['train'].patient_id)&set(frames['validation'].patient_id):raise ValueError('Overlap pasien train-validation')
    if set(frames['train'].sample_id)&set(frames['validation'].sample_id):raise ValueError('Overlap citra train-validation')
    return frames,candidates[0]

def analyze_dataset(root,research,preprocessing_root,out,require_cache=True):
    root=Path(root);out=Path(out);out.mkdir(parents=True,exist_ok=True)
    protocol=read_json(root/'protocol.json');plan=read_json(root/'run_plan.json');ds=protocol['dataset']
    if protocol['threshold']!=.5:raise ValueError('Notebook ini mengaudit threshold tetap 0.5')
    for name,key in [('b05_hinton_engine.py','engine_sha256'),('b04_compat.py','b04_compat_sha256')]:
        if hashlib.sha256(read_bytes(root/name)).hexdigest()!=protocol[key]:raise ValueError('Kode sumber berubah: '+name)
    expected={(s,t) for s in ['mobilenet_v2','nasnet_mobile'] for t in ['none','vgg16','densenet121']}
    if set(map(tuple,plan['runs']))!=expected or len(plan['runs'])!=6:raise ValueError('Rencana run bukan enam kombinasi')
    summary=read_csv(root/'validation_summary.csv');split=read_csv(root/'split_audit.csv')
    if len(summary)!=6:raise ValueError('Ringkasan belum lengkap')
    missing=[]
    try:frames,ready=ready_frames(protocol,preprocessing_root)
    except FileNotFoundError as e:
        if require_cache:raise
        missing.append(str(e));frames=None;ready=None
    pred_rows=[];history_rows=[];stage_rows=[];loss_rows=[];inits={};reference=None
    fig,axes=plt.subplots(6,2,figsize=(12,22),squeeze=False)
    for i,(student,teacher) in enumerate(plan['runs']):
        prefix=root/(student+'__from__'+teacher)/('seed_'+str(protocol['training_seed']))
        cfg=read_json(prefix/'config.json');complete=read_json(prefix/'COMPLETE.json')
        if hashlib.sha256(read_bytes(prefix/'config.json')).hexdigest()!=complete['config_sha256']:raise ValueError('Config/COMPLETE mismatch')
        for k,v in protocol.items():
            if cfg.get(k)!=v:raise ValueError('Run/protocol mismatch: '+k)
        init=read_json(prefix/'head/initialization.json')['weights_sha256'];inits.setdefault(student,set()).add(init)
        p=read_csv(prefix/'validation_predictions.csv');check_table(p)
        if reference is None:reference=frames['validation'] if frames is not None else p[['sample_id','patient_id','label']]
        p=align(p,reference);m=measures(p.label,p.probability)
        if not np.array_equal(p.prediction,(p.probability>=.5).astype(int)):raise ValueError('Prediksi bukan threshold 0.5')
        saved=read_json(prefix/'validation_metrics.json')
        row=summary[(summary.model==student)&(summary.teacher==teacher)]
        if len(row)!=1:raise ValueError('Ringkasan model duplikat/hilang')
        for k,v in m.items():
            np.testing.assert_allclose(v,saved[k],atol=1e-12,rtol=0)
            np.testing.assert_allclose(v,row.iloc[0][k],atol=1e-12,rtol=0)
        p=p.copy();p['model']=student;p['teacher']=teacher;p['dataset']=ds;pred_rows.append(p)
        history=read_csv(prefix/'history.csv');alpha=float(protocol['alpha']) if teacher!='none' else 0.
        if not np.isfinite(history[['loss','supervised_loss','distillation_loss_T2']]).all().all():raise ValueError('Nonfinite loss')
        history['weighted_supervised']=(1-alpha)*history.supervised_loss
        history['weighted_KD']=alpha*history.distillation_loss_T2
        np.testing.assert_allclose(history.loss,history.weighted_supervised+history.weighted_KD,atol=1e-6)
        history['dataset']=ds;history['model']=student;history['teacher']=teacher;history_rows.append(history)
        stage=read_json(prefix/'stage_comparison.json');h=stage['head'];f=stage['finetune']
        selected='finetune' if (f['macro_f1'],f['recall'])>(h['macro_f1'],h['recall']) else 'head'
        if selected!=complete['best_stage'] or selected!=stage['selected']:raise ValueError('Seleksi stage tidak cocok')
        for phase in ['head','finetune']:
            ph=history[history.stage==phase]
            top=ph.sort_values(['macro_f1','recall','phase_epoch'],ascending=[False,False,True]).iloc[0]
            if int(top.phase_epoch)!=int(stage[phase]['best_phase_epoch']):raise ValueError('Seleksi epoch tidak cocok')
        stage_rows.append(dict(dataset=ds,model=student,teacher=teacher,selected=selected,
            selected_epoch=saved['best_epoch'],head_macro_f1=h['macro_f1'],finetune_macro_f1=f['macro_f1'],
            finetune_epochs=int((history.stage=='finetune').sum()),all_negative=bool((p.prediction==0).all()),
            probability_max=float(p.probability.max()),recall=m['recall'],f1=m['f1'],macro_f1=m['macro_f1'],roc_auc=m['roc_auc']))
        for phase in ['head','finetune']:
            ph=history[history.stage==phase]
            loss_rows.append(dict(dataset=ds,model=student,teacher=teacher,stage=phase,
                mean_weighted_supervised=ph.weighted_supervised.mean(),mean_weighted_KD=ph.weighted_KD.mean(),
                first_loss=ph.loss.iloc[0],last_loss=ph.loss.iloc[-1],epochs=len(ph),
                zero_recall_epochs=int((ph.recall==0).sum())))
        ax=axes[i,0];ax.plot(history.global_epoch,history.weighted_supervised,label='(1-alpha) supervised')
        ax.plot(history.global_epoch,history.weighted_KD,label='alpha KD (includes T²)')
        ax.axvline(protocol['head_epochs']+.5,color='gray',linestyle=':');ax.set_title(student+' <- '+teacher);ax.legend(fontsize=8)
        ax.set_xlabel('Epoch');ax.set_ylabel('Mean training loss contribution')
        ax=axes[i,1];ax.plot(history.global_epoch,history.macro_f1,label='Validation macro-F1')
        ax.plot(history.global_epoch,history.recall,label='Validation recall');ax.plot(history.global_epoch,history.roc_auc,label='Validation AUC')
        ax.axvline(saved['best_epoch'],color='black',linestyle=':',label='Selected epoch');ax.set_ylim(-.02,1.02);ax.legend(fontsize=8)
        ax.set_xlabel('Epoch');ax.set_title('Validation; threshold 0.5')
    if any(len(x)!=1 for x in inits.values()):raise ValueError('Initial weights differ')
    fig.suptitle(ds+' — existing B05 history',fontsize=14);fig.tight_layout(rect=[0,0,1,.98]);fig.savefig(out/'history_diagnostics.png',dpi=120);plt.close(fig)
    pd.concat(pred_rows,ignore_index=True).to_csv(out/'validation_student_predictions.csv',index=False)
    pd.concat(history_rows,ignore_index=True).to_csv(out/'history_loss_contributions.csv',index=False)
    pd.DataFrame(stage_rows).to_csv(out/'checkpoint_diagnostics.csv',index=False)
    pd.DataFrame(loss_rows).to_csv(out/'loss_summary.csv',index=False)
    teacher_rows=[];grad_rows=[];flip_rows=[];cache_rows=[];bound_rows=[]
    for teacher in ['vgg16','densenet121']:
        source=protocol['teacher_sources'][teacher];cache=mapped_path(plan['teacher_cache_folders'][teacher],research)
        if not cache.exists():
            if require_cache:raise FileNotFoundError('Teacher cache tidak ditemukan: '+str(cache))
            missing.append('Missing teacher cache: '+str(cache));continue
        manifest=read_json(cache/'manifest.json');done=read_json(cache/'COMPLETE.json');audit=read_json(cache/'teacher_audit.json')
        if not audit.get('passed'):raise ValueError('Teacher audit gagal')
        if manifest['source']!=source or manifest['data_sha256']!=protocol['data_sha256']:raise ValueError('Cache source/data berbeda')
        if manifest['builder_sha256']!=protocol['engine_sha256']:raise ValueError('Cache engine berbeda')
        digest=protocol['teacher_cache_sha256'][teacher]
        if done['csv_sha256']!=digest:raise ValueError('Cache COMPLETE hash berbeda')
        table=read_csv(cache/'train_logits.csv',digest)
        if frames is None:raise ValueError('Frozen train CSV diperlukan untuk audit cache')
        table=align(table,frames['train']);y=table.label.to_numpy(dtype=int)
        if not np.isfinite(table[['logit_original','logit_flipped']]).all().all():raise ValueError('Logit tidak finite')
        counts=table.label.value_counts();cw={k:len(table)/(2*int(counts[k])) for k in [0,1]}
        alpha=float(protocol['alpha']);temp=float(protocol['temperature'])
        # Upper/lower bounds for ANY Teacher target q_T in [0,1], at Student z=0.
        for label in [0,1]:
            hard=(1-alpha)*cw[label]*(.5-label)
            bound_rows.append(dict(dataset=ds,teacher=teacher,label=label,class_weight=cw[label],
                weighted_hard_gradient_at_z0=hard,total_gradient_min=hard-alpha*temp*.5,
                total_gradient_max=hard+alpha*temp*.5,
                definition='dL/dStudent_logit per image before batch averaging; reference z=0'))
        for view,column in [('original','logit_original'),('horizontal_flip','logit_flipped')]:
            z=table[column].to_numpy(dtype=float);prob=sigmoid(z);q=sigmoid(z/temp)
            per=table[['sample_id','patient_id','label']].copy();per['dataset']=ds;per['teacher']=teacher;per['view']=view
            per['logit']=z;per['probability_T1']=prob;per['soft_target_T']=q;cache_rows.append(per)
            for label in [0,1]:
                mask=y==label
                r=dict(dataset=ds,teacher=teacher,view=view,label=label,class_weight=cw[label])
                for name,values in [('logit',z[mask]),('p_T1',prob[mask]),('q_T',q[mask])]:
                    r.update({name+'_'+k:v for k,v in distribution(values).items()})
                r['teacher_error_fraction_T1']=float(np.mean((prob[mask]>=.5)!=y[mask]));teacher_rows.append(r)
            for p_reference in [.25,.5,.75]:
                zs=np.full(len(y),np.log(p_reference/(1-p_reference)))
                gh,gs=gradient_terms(y,zs,z,cw,alpha,temp);gt=gh+gs
                for label in [0,1]:
                    mask=y==label;desired=1 if label==0 else -1
                    grad_rows.append(dict(dataset=ds,teacher=teacher,view=view,label=label,student_p_reference=p_reference,
                        mean_hard_gradient=float(gh[mask].mean()),mean_soft_gradient=float(gs[mask].mean()),
                        mean_total_gradient=float(gt[mask].mean()),fraction_hard_soft_opposed=float(np.mean(gh[mask]*gs[mask]<0)),
                        fraction_total_points_away_from_label=float(np.mean(gt[mask]*desired<0)),
                        note='Hypothetical scalar logit gradients; NOT measured network/training gradients'))
        po=sigmoid(table.logit_original);pf=sigmoid(table.logit_flipped)
        for label in [0,1]:
            mask=y==label
            flip_rows.append(dict(dataset=ds,teacher=teacher,label=label,n=int(mask.sum()),
                mean_abs_probability_change=float(np.abs(po-pf)[mask].mean()),
                max_abs_probability_change=float(np.abs(po-pf)[mask].max()),
                prediction_change_fraction=float(np.mean((po[mask]>=.5)!=(pf[mask]>=.5)))))
        valpath=mapped_path(source['folder'],research)/'validation_predictions.csv'
        if not valpath.exists():
            if require_cache:raise FileNotFoundError('Prediksi validation Teacher tidak ditemukan: '+str(valpath))
            missing.append('Missing Teacher validation predictions: '+str(valpath))
        else:
            tv=align(read_csv(valpath,source['validation_predictions_sha256']),reference)
            for k,v in measures(tv.label,tv.probability).items():np.testing.assert_allclose(v,audit['current_validation_metrics'][k],atol=1e-12)
            tv.to_csv(out/(teacher+'_validation_predictions.csv'),index=False)
    for name,rows in [('teacher_train_distributions',teacher_rows),('gradient_reference_diagnostics',grad_rows),
                      ('flip_consistency',flip_rows),('boundary_gradient_bounds',bound_rows)]:
        if rows:pd.DataFrame(rows).to_csv(out/(name+'.csv'),index=False)
    if cache_rows:
        cat=pd.concat(cache_rows,ignore_index=True);cat.to_csv(out/'teacher_train_logits_and_targets.csv',index=False)
        fig,axes=plt.subplots(2,2,figsize=(11,8))
        for ax,(teacher,view) in zip(axes.flat,[(t,v) for t in ['vgg16','densenet121'] for v in ['original','horizontal_flip']]):
            sub=cat[(cat.teacher==teacher)&(cat.view==view)]
            for label in [0,1]:ax.hist(sub[sub.label==label].soft_target_T,bins=np.linspace(0,1,21),alpha=.55,label='Label '+str(label))
            ax.set_title(teacher+' '+view);ax.set_xlabel('Teacher soft target q_T');ax.set_ylabel('Images');ax.legend()
        fig.tight_layout();fig.savefig(out/'teacher_soft_targets.png',dpi=140);plt.close(fig)
    status=dict(dataset=ds,complete=not missing,missing=missing,test_predictions_read=False,training_performed=False,
                original_protocol_unchanged=True,ready_path=str(ready) if ready else None,
                all_negative_runs=[r['model']+' <- '+r['teacher'] for r in stage_rows if r['all_negative']],
                limitations=['No Teacher cache recomputation or GPU checkpoint inference.',
                    'Reference gradients use hypothetical Student probabilities, not actual training logits or network-parameter gradients.',
                    'Teacher train metrics are in-sample and not evidence of generalization.',
                    'Passing diagnosis does not automatically freeze or approve the research protocol.'])
    save_json(out/'diagnosis_status.json',status)
    save_json(out/'source_protocol.json',protocol)
    return pd.DataFrame(stage_rows),pd.DataFrame(loss_rows),pd.DataFrame(teacher_rows),pd.DataFrame(bound_rows),status


## Cell 3 — Pemeriksaan matematis gradien

Loss B05 adalah $(1-\alpha)w_y\,BCE(y,\sigma(z_s))+\alpha T^2 KL(q_t\Vert q_s)$, dengan $q_t=\sigma(z_t/T)$.

Turunan per citra terhadap **logit Student** (sebelum pembagian dengan jumlah citra dalam batch) adalah:

$$g_{sup}=(1-\alpha)w_y(\sigma(z_s)-y),\qquad g_{KD}=\alpha T(\sigma(z_s/T)-q_t).$$

Gradien positif menurunkan logit saat gradient descent; gradien negatif menaikkannya. Notebook mengevaluasi titik referensi probabilitas Student 0,25; 0,5; dan 0,75. Ini **diagnosis analitis pada titik hipotetis**, bukan rekaman gradien aktual selama training, dan bukan gradien terhadap bobot jaringan. Loss scalar tidak cukup untuk membuktikan dominasi gradien.

Cell berikut menguji turunan analitis terhadap finite differences. Tidak ada tuning atau pemilihan threshold baru.

In [ ]:
def scalar_loss(y,zs,zt,cw,alpha,temp):
    hard=cw[int(y)]*(np.logaddexp(0.,zs)-y*zs)
    a=zt/temp;b=zs/temp;q=float(sigmoid(a))
    kl=q*(-np.logaddexp(0.,-a)+np.logaddexp(0.,-b))+(1-q)*(-np.logaddexp(0.,a)+np.logaddexp(0.,b))
    return (1-alpha)*hard+alpha*temp*temp*kl
for label in [0,1]:
    for zs in [-1.2,0.,1.2]:
        for zt in [-4.,.3,4.]:
            cw={0:.61,1:2.77};eps=1e-5
            gh,gk=gradient_terms(np.array([label]),np.array([zs]),np.array([zt]),cw,.5,2.)
            fd=(scalar_loss(label,zs+eps,zt,cw,.5,2.)-scalar_loss(label,zs-eps,zt,cw,.5,2.))/(2*eps)
            np.testing.assert_allclose(gh[0]+gk[0],fd,rtol=1e-6,atol=1e-8)
print('Turunan loss sesuai finite differences.')


## Cell 4 — Jalankan diagnosis empat dataset
Membaca `train_logits.csv` untuk original dan horizontal flip serta prediksi validation. Tidak melakukan inferensi ulang. Perbandingan target per kelas merupakan deskripsi training; tidak dianggap performa pada data baru.

Pada gradien referensi, `fraction_total_points_away_from_label` menunjukkan proporsi arah turunan logit yang berlawanan dengan label pada titik referensi tersebut. Angka ini tidak mengukur konflik gradien parameter jaringan. Perbedaan output original–flip merupakan sensitivitas Teacher terhadap augmentasi; bukan otomatis kesalahan cache.

In [ ]:
import traceback
RESULTS={};STATUSES=[]
for dataset in DATASETS:
    print('\nDIAGNOSIS',dataset,flush=True)
    destination=OUTPUT/dataset
    try:
        result=analyze_dataset(B05_ROOT/dataset/EXPERIMENT_IDS[dataset],RESEARCH,
                               PREPROCESSING_ROOT,destination,require_cache=True)
        RESULTS[dataset]=result;STATUSES.append(result[-1])
        display(result[0].round(4))
    except Exception as error:
        destination.mkdir(parents=True,exist_ok=True)
        status=dict(dataset=dataset,complete=False,error_type=type(error).__name__,error=str(error),
                    test_predictions_read=False,training_performed=False)
        STATUSES.append(status);save_json(destination/'diagnosis_status.json',status)
        (destination/'error_traceback.txt').write_text(traceback.format_exc(),encoding='utf-8')
        print('BELUM LENGKAP:',type(error).__name__,str(error),flush=True)
print('\nLanjutkan sampai Cell 6 untuk mengunduh laporan, termasuk bila ada error.')


## Cell 5 — Ringkasan dan batas interpretasi

Periksa berurutan:
1. **Integritas:** semua dataset berstatus lengkap; sumber/cache/ID tidak berubah.
2. **Teacher:** distribusi target per kelas; jumlah kesalahan pada training; perubahan original–flip.
3. **Student:** prediksi validation semua negatif atau tidak; loss dan Recall sepanjang epoch.
4. **Seleksi:** apakah fine-tuning membaik, berhenti saat metrik stagnan, dan checkpoint awal terpilih karena seri.
5. **Diagnosis analitis:** arah kontribusi supervised dan KD pada probabilitas referensi. Tidak menyimpulkan kausalitas hanya dari satu statistik.

Pada INbreast training terdapat 42 positif dari 233 citra. Untuk konfigurasi B05 saat ini, bobot kelas positif adalah 233/(2×42). Pada $z_s=0$, turunan total per citra positif tetap negatif bahkan untuk target Teacher paling rendah. Ini membatasi dugaan bahwa target Teacher sendirian membalik arah gradien kelas positif di batas 0,5; tidak menjamin jaringan mampu mempelajari representasi yang tepat.

Pemeriksaan ini tidak mengubah warning identitas lama di protokol yang sudah di-hash. Audit identitas INbreast yang dilakukan terpisah tetap menjadi dokumen pendukung.

In [ ]:
display(pd.DataFrame([dict(dataset=s['dataset'],complete=s['complete'],
    all_negative_runs='; '.join(s.get('all_negative_runs',[])),error=s.get('error','')) for s in STATUSES]))
if 'inbreast' in RESULTS:
    checkpoints,losses,targets,bounds,status=RESULTS['inbreast']
    print('INbreast — loss aktual dari histori:')
    display(losses.round(5))
    print('INbreast — distribusi target Teacher per kelas:')
    display(targets[['teacher','view','label','q_T_mean','q_T_median','teacher_error_fraction_T1']].round(5))
    print('INbreast — batas gradien untuk SEMUA kemungkinan target Teacher di z_student=0:')
    display(bounds[['teacher','label','class_weight','total_gradient_min','total_gradient_max']].round(6))
print('Status lengkap berarti diagnosis selesai; bukan persetujuan otomatis untuk membuka test.')


## Cell 6 — Simpan dan unduh ZIP diagnosis
Kirim ZIP ini untuk ditinjau. ZIP memuat tabel target Teacher training, sensitivitas flip, diagnosis gradien referensi, prediksi validation, histori loss, grafik, status, dan hash input. Tidak memuat citra atau checkpoint. Pertahankan `EVALUATE_TEST=False` di B05 selama diagnosis dan keputusan konfigurasi belum selesai.

In [ ]:
save_json(OUTPUT/'diagnosis_summary.json',dict(datasets=STATUSES,
    all_complete=bool(STATUSES) and all(s['complete'] for s in STATUSES),
    evaluation_test=False,training_performed=False,python=platform.python_version(),
    numpy=np.__version__,pandas=pd.__version__))
pd.DataFrame(READ_LEDGER).drop_duplicates().to_csv(OUTPUT/'input_hash_audit.csv',index=False)
(OUTPUT/'READ_ME.txt').write_text(
    'B05A diagnosis only. No training, checkpoint changes, threshold optimization or test evaluation.\n'
    'Gradient tables use hypothetical Student logits, not measured training gradients.\n'
    'Loss magnitudes alone do not establish gradient dominance. Teacher training outputs are in-sample.\n'
    'Read diagnosis_summary.json for completeness/errors before interpreting the report.\n',encoding='utf-8')
REPORT=OUTPUT.parent/('B05A_KD_diagnostics_'+STAMP+'.zip')
with zipfile.ZipFile(REPORT,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file():archive.write(path,path.relative_to(OUTPUT))
print('Laporan:',REPORT)
print('Semua lengkap:',all(s['complete'] for s in STATUSES))
if DOWNLOAD_REPORT:files.download(str(REPORT))
